# Data Quality Checks (Round 2)

Round 1 cleaning already happened in `ingest.py` (duplicates dropped, types fixed).
This notebook is the closer look: nulls, outliers, and sanity checks on the data
now sitting in Postgres.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import sys
sys.path.append('../src')
from config import SQLALCHEMY_URL

engine = create_engine(SQLALCHEMY_URL)
print('connected')

## 1. Load each table into a DataFrame

We pull the data back OUT of Postgres into pandas so we can inspect it easily.

In [ ]:
customers = pd.read_sql('SELECT * FROM dim_customers', engine)
products = pd.read_sql('SELECT * FROM dim_products', engine)
sellers = pd.read_sql('SELECT * FROM dim_sellers', engine)
orders = pd.read_sql('SELECT * FROM dim_orders', engine)
order_items = pd.read_sql('SELECT * FROM fact_order_items', engine)
payments = pd.read_sql('SELECT * FROM fact_payments', engine)
reviews = pd.read_sql('SELECT * FROM fact_reviews', engine)

print('customers:', customers.shape)
print('products:', products.shape)
print('sellers:', sellers.shape)
print('orders:', orders.shape)
print('order_items:', order_items.shape)
print('payments:', payments.shape)
print('reviews:', reviews.shape)

`.shape` tells you (rows, columns). This is your first sanity check —
do these row counts roughly match what you saw printed when `ingest.py` ran?

## 2. Check for missing values (nulls)

`.isnull().sum()` counts how many empty/missing values are in each column.

In [ ]:
print('--- orders nulls ---')
print(orders.isnull().sum())

Some nulls here are EXPECTED, not errors. For example, `order_delivered_customer_date`
will be null for orders that were cancelled or never delivered — that's real, not broken data.
The question to ask is: does the *number* of nulls make sense given the `order_status` column?

In [ ]:
# Does every 'delivered' order actually have a delivery date?
delivered = orders[orders['order_status'] == 'delivered']
missing_delivery_date = delivered['order_delivered_customer_date'].isnull().sum()
print(f"Orders marked 'delivered' but missing a delivery date: {missing_delivery_date}")

If that number is 0, great — the data is consistent. If it's not 0, that's a real data
quality issue worth writing down (e.g. "X orders had inconsistent status/date data,
excluded from delivery-time analysis").

## 3. Check for outliers in price

`.describe()` gives you min, max, average, and percentiles — a fast way to spot
something that looks broken (like a price of 0 or an impossibly huge number).

In [ ]:
print(order_items['price'].describe())
print()
print('Items priced at 0 or less:', (order_items['price'] <= 0).sum())
print('Top 5 highest priced items:')
print(order_items.sort_values('price', ascending=False).head(5)[['product_id', 'price']])

## 4. Duplicate check at the business level

Row-level duplicates were already removed in `ingest.py`. Here we check something
trickier: does one real-world customer somehow show up as more than one
`customer_unique_id`? (Common in real e-commerce data — same person, multiple accounts.)

In [ ]:
dupe_check = customers.groupby('customer_unique_id')['customer_id'].nunique()
print('Customers with more than one customer_id:', (dupe_check > 1).sum())

## 5. Foreign key sanity check

The database should already enforce this (it wouldn't have let bad data in), but
it's good practice to confirm: does every `order_id` in `fact_order_items` actually
exist in `dim_orders`?

In [ ]:
orphan_items = ~order_items['order_id'].isin(orders['order_id'])
print('Order items with no matching order:', orphan_items.sum())

## Summary

Write your findings here once you've run all the cells above, e.g.:
- Row counts matched expectations: yes/no
- Null patterns made sense: yes/no, notes
- Outliers found: none / list them
- Duplicate customers found: X
- Orphan records found: X

This becomes the "data cleaning" section of your project writeup.